In [1]:
# TAHAP 1A PERSIAPAN LIBRARY DAN KONFIGURASI API

import requests
import pandas as pd
from io import StringIO

print("Library berhasil diimport.")

# Konfigurasi API e-Stat
APP_ID = "69fbbb74fa35bfa86664ac12004c7b8b7296ea15"
STATS_DATA_ID = "0000010212"

print("Konfigurasi API berhasil disiapkan.")
print(f"Stats Data ID : {STATS_DATA_ID}")

Library berhasil diimport.
Konfigurasi API berhasil disiapkan.
Stats Data ID : 0000010212


In [2]:
# TAHAP 1B PEMBENTUKAN URL REQUEST API

print("Mulai membentuk URL API e-Stat...")

# Membentuk endpoint API
url = f"""
http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData
?appId={APP_ID}
&lang=J
&statsDataId={STATS_DATA_ID}
&metaGetFlg=Y
&cntGetFlg=N
&explanationGetFlg=Y
&annotationGetFlg=Y
&sectionHeaderFlg=1
&replaceSpChars=0
"""

# Membersihkan format URL
url = url.replace("\n", "").replace(" ", "")

print("URL API berhasil dibentuk.")

Mulai membentuk URL API e-Stat...
URL API berhasil dibentuk.


In [3]:
# TAHAP 1C PENGIRIMAN REQUEST KE API e-STAT

print("Mulai menghubungkan ke API e-Stat...")

# Mengirim request ke API
response = requests.get(url)

print(f"Status koneksi API : {response.status_code}")

if response.status_code == 200:
    print("Koneksi API berhasil.")
else:
    print("Koneksi API gagal.")

Mulai menghubungkan ke API e-Stat...
Status koneksi API : 200
Koneksi API berhasil.


In [4]:
# TAHAP 1D PEMBACAAN STRUKTUR DATA RESPON API

print("Mulai membaca struktur data dari API...")

# Memisahkan isi response menjadi baris-baris teks
lines = response.text.splitlines()

# Mencari posisi awal tabel data asli
value_index = None

for i, line in enumerate(lines):
    if line.strip().replace('"', '') == "VALUE":
        value_index = i
        break

if value_index is not None:
    print("Penanda awal data berhasil ditemukan.")
    print(f"Posisi baris VALUE ditemukan pada index ke-{value_index}")
else:
    print("Penanda VALUE tidak ditemukan.")

Mulai membaca struktur data dari API...
Penanda awal data berhasil ditemukan.
Posisi baris VALUE ditemukan pada index ke-28


In [5]:
# TAHAP 1E MEMUAT DATA KE DATAFRAME PANDAS

print("Mulai memuat data mentah ke DataFrame...")

# Mengambil isi CSV setelah baris VALUE
csv_text = "\n".join(lines[value_index + 1:])

# Memuat data ke DataFrame
df_biaya_hidup_mentah = pd.read_csv(StringIO(csv_text))

# --- TAMBAHAN PATH TARGET UNTUK DATA MENTAH BIAYA HIDUP ---
path_raw_biaya_hidup = "../data/raw/raw_biaya_hidup_japan.csv"

# Mengekspor data mentah asli dari API ke folder raw
df_biaya_hidup_mentah.to_csv(path_raw_biaya_hidup, index=False, encoding='utf-8')
# ---------------------------------------------------------

print("\n=============================================")
print("        TAHAP 1 SELESAI (BIAYA HIDUP)        ")
print("=============================================")

print("Data mentah berhasil dimuat ke DataFrame dan disimpan fisik.")
print(f"Lokasi penyimpanan mentah: {path_raw_biaya_hidup}")
print(f"Total data awal : {df_biaya_hidup_mentah.shape[0]} baris")
print(f"Total kolom     : {df_biaya_hidup_mentah.shape[1]} kolom")

print("\n--- Daftar Kolom Dataset ---")
print(df_biaya_hidup_mentah.columns.tolist())

Mulai memuat data mentah ke DataFrame...

        TAHAP 1 SELESAI (BIAYA HIDUP)        
Data mentah berhasil dimuat ke DataFrame dan disimpan fisik.
Lokasi penyimpanan mentah: ../data/raw/raw_biaya_hidup_japan.csv
Total data awal : 85104 baris
Total kolom     : 11 kolom

--- Daftar Kolom Dataset ---
['tab_code', '観測値', 'cat01_code', 'Ｌ\u3000家計', 'area_code', '地域', 'time_code', '調査年', 'unit', 'value', 'annotation']


In [6]:
# TAHAP 2A PERSIAPAN DATA UNTUK PROSES FILTERING

print("Mulai menyiapkan data biaya hidup...")

# Membuat salinan data mentah
df_bh_filter = df_biaya_hidup_mentah.copy()

print("Data berhasil disiapkan.")
print(f"Total data awal : {df_bh_filter.shape[0]} baris")

Mulai menyiapkan data biaya hidup...
Data berhasil disiapkan.
Total data awal : 85104 baris


In [7]:
# TAHAP 2B FILTER DATA NASIONAL

print("Mulai menghapus data nasional...")

# Menghapus data nasional Jepang
df_bh_filter = df_bh_filter[df_bh_filter['地域'] != '全国']

print("Filter data nasional berhasil dilakukan.")
print(f"Total data setelah filter wilayah : {df_bh_filter.shape[0]} baris")

Mulai menghapus data nasional...
Filter data nasional berhasil dilakukan.
Total data setelah filter wilayah : 83331 baris


In [8]:
# TAHAP 2C FILTER INDIKATOR BIAYA HIDUP

print("Mulai menyaring indikator biaya hidup...")

# Kode indikator pengeluaran konsumsi bulanan
KODE_BIAYA_HIDUP = "#L02211"

# Mengambil hanya indikator biaya hidup
df_bh_filter = df_bh_filter[
    df_bh_filter['cat01_code'] == KODE_BIAYA_HIDUP
]

print("Filter indikator berhasil dilakukan.")
print(f"Indikator yang digunakan : {KODE_BIAYA_HIDUP}")
print(f"Total data saat ini : {df_bh_filter.shape[0]} baris")

Mulai menyaring indikator biaya hidup...
Filter indikator berhasil dilakukan.
Indikator yang digunakan : #L02211
Total data saat ini : 1175 baris


In [9]:
# TAHAP 2D FILTER RENTANG TAHUN PENELITIAN

print("Mulai menyaring data berdasarkan tahun penelitian...")

# Menentukan tahun target penelitian
tahun_target_bh = [
    '2020年度',
    '2021年度',
    '2022年度',
    '2023年度',
    '2024年度',
    '2025年度',
    '2026年度'
]

# Mengambil data sesuai tahun penelitian
df_bh_filter = df_bh_filter[
    df_bh_filter['調査年'].isin(tahun_target_bh)
]

print("Filter tahun penelitian berhasil dilakukan.")
print(f"Tahun yang digunakan : {tahun_target_bh}")

print("\nTahun yang terdeteksi di dataset:")
print(df_bh_filter['調査年'].unique().tolist())

print(f"\nTotal data hasil filtering : {df_bh_filter.shape[0]} baris")

Mulai menyaring data berdasarkan tahun penelitian...
Filter tahun penelitian berhasil dilakukan.
Tahun yang digunakan : ['2020年度', '2021年度', '2022年度', '2023年度', '2024年度', '2025年度', '2026年度']

Tahun yang terdeteksi di dataset:
['2020年度', '2021年度', '2022年度', '2023年度', '2024年度']

Total data hasil filtering : 235 baris


In [10]:
# TAHAP 2E VERIFIKASI HASIL FILTERING DATA

print("\n=============================================")
print("        TAHAP 2 SELESAI (BIAYA HIDUP)        ")
print("=============================================")

print("Proses filtering data biaya hidup berhasil dilakukan.")

print("\n--- Sampel Data Hasil Filtering ---")

kolom_tampil = [
    '地域',
    '調査年',
    'unit',
    'value'
]

print(df_bh_filter[kolom_tampil].head(5))


        TAHAP 2 SELESAI (BIAYA HIDUP)        
Proses filtering data biaya hidup berhasil dilakukan.

--- Sampel Data Hasil Filtering ---
       地域     調査年 unit  value
8445  北海道  2020年度   千円  301.7
8446  北海道  2021年度   千円  268.4
8447  北海道  2022年度   千円  277.7
8448  北海道  2023年度   千円  296.9
8449  北海道  2024年度   千円  285.5


In [11]:
# TAHAP 3A PERSIAPAN DATA UNTUK PROSES CLEANING

import numpy as np

print("Mulai menyiapkan data untuk proses pembersihan...")

# Membuat salinan data hasil filtering
df_bh_clean = df_bh_filter.copy()

print("Data berhasil disiapkan.")
print(f"Total data awal : {df_bh_clean.shape[0]} baris")

Mulai menyiapkan data untuk proses pembersihan...
Data berhasil disiapkan.
Total data awal : 235 baris


In [12]:
# TAHAP 3B PEMBERSIHAN NILAI KOSONG PADA KOLOM VALUE

print("Mulai membersihkan nilai kosong pada kolom value...")

# Menghapus spasi tersembunyi
df_bh_clean['value'] = df_bh_clean['value'].astype(str).str.strip()

# Mengganti simbol kosong menjadi NaN
df_bh_clean['value'] = df_bh_clean['value'].replace(
    ['-', '‐', ''],
    np.nan
)

print("Pembersihan nilai kosong berhasil dilakukan.")

Mulai membersihkan nilai kosong pada kolom value...
Pembersihan nilai kosong berhasil dilakukan.


In [13]:
# TAHAP 3C PENGHAPUSAN BARIS DATA KOSONG

print("Mulai menghapus baris yang memiliki nilai kosong...")

# Menghapus baris yang memiliki NaN pada kolom value
df_bh_clean = df_bh_clean.dropna(subset=['value'])

print("Baris data kosong berhasil dihapus.")
print(f"Total data setelah penghapusan : {df_bh_clean.shape[0]} baris")

Mulai menghapus baris yang memiliki nilai kosong...
Baris data kosong berhasil dihapus.
Total data setelah penghapusan : 235 baris


In [14]:
# TAHAP 3D KONVERSI TIPE DATA VALUE

print("Mulai mengubah tipe data kolom value...")

# Mengubah tipe data value menjadi float
df_bh_clean['value'] = df_bh_clean['value'].astype(float)

print("Konversi tipe data berhasil dilakukan.")
print("Kolom value berhasil diubah menjadi float.")

Mulai mengubah tipe data kolom value...
Konversi tipe data berhasil dilakukan.
Kolom value berhasil diubah menjadi float.


In [15]:
# TAHAP 3E VERIFIKASI HASIL PEMBERSIHAN DATA

print("\n=============================================")
print("        TAHAP 3 SELESAI (BIAYA HIDUP)        ")
print("=============================================")

print("Pembersihan data dan konversi tipe data berhasil dilakukan.")

print("\n--- Verifikasi Struktur Kolom Value ---")

print(df_bh_clean[['value']].info())


        TAHAP 3 SELESAI (BIAYA HIDUP)        
Pembersihan data dan konversi tipe data berhasil dilakukan.

--- Verifikasi Struktur Kolom Value ---
<class 'pandas.core.frame.DataFrame'>
Index: 235 entries, 8445 to 9599
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   value   235 non-null    float64
dtypes: float64(1)
memory usage: 3.7 KB
None


In [16]:
# TAHAP 4A PERSIAPAN DATA KONVERSI NOMINAL

print("Mulai menyiapkan data untuk konversi nominal...")

# Membuat salinan data hasil cleaning
df_bh_nominal = df_bh_clean.copy()

print("Data berhasil disiapkan.")
print(f"Total data awal : {df_bh_nominal.shape[0]} baris")

Mulai menyiapkan data untuk konversi nominal...
Data berhasil disiapkan.
Total data awal : 235 baris


In [17]:
# TAHAP 4B KONVERSI NILAI BIAYA HIDUP KE YEN ASLI

print("Mulai mengubah nilai biaya hidup ke nominal Yen asli...")

# Mengubah nilai ribuan Yen menjadi nominal Yen asli
df_bh_nominal['biaya_hidup_nominal_yen'] = (
    df_bh_nominal['value'] * 1000
).astype('int64')

print("Konversi nominal Yen berhasil dilakukan.")

Mulai mengubah nilai biaya hidup ke nominal Yen asli...
Konversi nominal Yen berhasil dilakukan.


In [18]:
# TAHAP 4C VERIFIKASI HASIL KONVERSI NOMINAL

print("\n=============================================")
print("        TAHAP 4 SELESAI (BIAYA HIDUP)        ")
print("=============================================")

print("Pembuatan kolom nominal biaya hidup berhasil dilakukan.")
print(f"Total data saat ini : {df_bh_nominal.shape[0]} baris")

print("\n--- Sampel Data Hasil Konversi Nominal ---")

kolom_tampil = [
    '地域',
    '調査年',
    'value',
    'biaya_hidup_nominal_yen'
]

print(df_bh_nominal[kolom_tampil].head(5))


        TAHAP 4 SELESAI (BIAYA HIDUP)        
Pembuatan kolom nominal biaya hidup berhasil dilakukan.
Total data saat ini : 235 baris

--- Sampel Data Hasil Konversi Nominal ---
       地域     調査年  value  biaya_hidup_nominal_yen
8445  北海道  2020年度  301.7                   301700
8446  北海道  2021年度  268.4                   268400
8447  北海道  2022年度  277.7                   277700
8448  北海道  2023年度  296.9                   296900
8449  北海道  2024年度  285.5                   285500


In [19]:
# TAHAP 5A PERSIAPAN DATA TRANSLASI

print("Mulai menyiapkan data untuk proses translasi...")

# Membuat salinan data hasil konversi nominal
df_bh_translate = df_bh_nominal.copy()

print("Data berhasil disiapkan.")
print(f"Total data awal : {df_bh_translate.shape[0]} baris")

Mulai menyiapkan data untuk proses translasi...
Data berhasil disiapkan.
Total data awal : 235 baris


In [20]:
# TAHAP 5B PEMBUATAN KAMUS TRANSLASI PREFEKTUR

print("Mulai membuat kamus translasi prefektur Jepang...")

# Kamus translasi prefektur Jepang
dict_prefektur = {
    '北海道': 'Hokkaido',
    '青森県': 'Aomori',
    '岩手県': 'Iwate',
    '宮城県': 'Miyagi',
    '秋田県': 'Akita',
    '山形県': 'Yamagata',
    '福島県': 'Fukushima',
    '茨城県': 'Ibaraki',
    '栃木県': 'Tochigi',
    '群馬県': 'Gunma',
    '埼玉県': 'Saitama',
    '千葉県': 'Chiba',
    '東京都': 'Tokyo',
    '神奈川県': 'Kanagawa',
    '新潟県': 'Niigata',
    '富山県': 'Toyama',
    '石川県': 'Ishikawa',
    '福井県': 'Fukui',
    '山梨県': 'Yamanashi',
    '長野県': 'Nagano',
    '岐阜県': 'Gifu',
    '静岡県': 'Shizuoka',
    '愛知県': 'Aichi',
    '三重県': 'Mie',
    '滋賀県': 'Shiga',
    '京都府': 'Kyoto',
    '大阪府': 'Osaka',
    '兵庫県': 'Hyogo',
    '奈良県': 'Nara',
    '和歌山県': 'Wakayama',
    '鳥取県': 'Tottori',
    '島根県': 'Shimane',
    '岡山県': 'Okayama',
    '広島県': 'Hiroshima',
    '山口県': 'Yamaguchi',
    '徳島県': 'Tokushima',
    '香川県': 'Kagawa',
    '愛媛県': 'Ehime',
    '高知県': 'Kochi',
    '福岡県': 'Fukuoka',
    '佐賀県': 'Saga',
    '長崎県': 'Nagasaki',
    '熊本県': 'Kumamoto',
    '大分県': 'Oita',
    '宮崎県': 'Miyazaki',
    '鹿児島県': 'Kagoshima',
    '沖縄県': 'Okinawa'
}

print("Kamus translasi berhasil dibuat.")
print(f"Total prefektur pada kamus : {len(dict_prefektur)} data")

Mulai membuat kamus translasi prefektur Jepang...
Kamus translasi berhasil dibuat.
Total prefektur pada kamus : 47 data


In [21]:
# TAHAP 5C TRANSLASI NAMA PREFEKTUR

print("Mulai menerjemahkan nama prefektur...")

# Mengubah nama prefektur Jepang menjadi Bahasa Inggris
df_bh_translate['prefektur_en'] = (
    df_bh_translate['地域']
    .map(dict_prefektur)
    .fillna(df_bh_translate['地域'])
)

print("Translasi prefektur berhasil dilakukan.")

Mulai menerjemahkan nama prefektur...
Translasi prefektur berhasil dilakukan.


In [22]:
# TAHAP 5D PENAMBAHAN LABEL KATEGORI BIAYA

print("Mulai menambahkan label kategori biaya...")

# Menambahkan label kategori biaya untuk kebutuhan sistem
df_bh_translate['kategori_biaya_id'] = 'Biaya Hidup Bulanan'

print("Label kategori biaya berhasil ditambahkan.")

Mulai menambahkan label kategori biaya...
Label kategori biaya berhasil ditambahkan.


In [23]:
# TAHAP 5E VERIFIKASI HASIL TRANSLASI

print("\n=============================================")
print("        TAHAP 5 SELESAI (BIAYA HIDUP)        ")
print("=============================================")

print("Penambahan translasi dan label kategori berhasil dilakukan.")
print(f"Total data saat ini : {df_bh_translate.shape[0]} baris")

print("\n--- Sampel Data Hasil Translasi ---")

kolom_tampil = [
    '地域',
    'prefektur_en',
    '調査年',
    'kategori_biaya_id',
    'biaya_hidup_nominal_yen'
]

print(df_bh_translate[kolom_tampil].head(5))


        TAHAP 5 SELESAI (BIAYA HIDUP)        
Penambahan translasi dan label kategori berhasil dilakukan.
Total data saat ini : 235 baris

--- Sampel Data Hasil Translasi ---
       地域 prefektur_en     調査年    kategori_biaya_id  biaya_hidup_nominal_yen
8445  北海道     Hokkaido  2020年度  Biaya Hidup Bulanan                   301700
8446  北海道     Hokkaido  2021年度  Biaya Hidup Bulanan                   268400
8447  北海道     Hokkaido  2022年度  Biaya Hidup Bulanan                   277700
8448  北海道     Hokkaido  2023年度  Biaya Hidup Bulanan                   296900
8449  北海道     Hokkaido  2024年度  Biaya Hidup Bulanan                   285500


In [24]:
# TAHAP 6A PERSIAPAN DATA ATRIBUT SAW

print("Mulai menyiapkan data untuk atribut SAW...")

# Membuat salinan data hasil translasi
df_bh_saw = df_bh_translate.copy()

print("Data berhasil disiapkan.")
print(f"Total data awal : {df_bh_saw.shape[0]} baris")

Mulai menyiapkan data untuk atribut SAW...
Data berhasil disiapkan.
Total data awal : 235 baris


In [25]:
# TAHAP 6B PENAMBAHAN ATRIBUT KRITERIA SAW

print("Mulai menambahkan atribut tipe kriteria...")

# Menambahkan tipe kriteria Cost
df_bh_saw['tipe_kriteria_biaya'] = 'Cost'

print("Atribut tipe kriteria berhasil ditambahkan.")

Mulai menambahkan atribut tipe kriteria...
Atribut tipe kriteria berhasil ditambahkan.


In [26]:
# TAHAP 6C VERIFIKASI HASIL ATRIBUT SAW

print("\n=============================================")
print("        TAHAP 6 SELESAI (BIAYA HIDUP)        ")
print("=============================================")

print("Penambahan atribut SAW berhasil dilakukan.")
print(f"Total data saat ini : {df_bh_saw.shape[0]} baris")

print("\n--- Sampel Data Hasil Atribut SAW ---")

kolom_tampil = [
    'prefektur_en',
    '調査年',
    'biaya_hidup_nominal_yen',
    'tipe_kriteria_biaya'
]

print(df_bh_saw[kolom_tampil].head(5))


        TAHAP 6 SELESAI (BIAYA HIDUP)        
Penambahan atribut SAW berhasil dilakukan.
Total data saat ini : 235 baris

--- Sampel Data Hasil Atribut SAW ---
     prefektur_en     調査年  biaya_hidup_nominal_yen tipe_kriteria_biaya
8445     Hokkaido  2020年度                   301700                Cost
8446     Hokkaido  2021年度                   268400                Cost
8447     Hokkaido  2022年度                   277700                Cost
8448     Hokkaido  2023年度                   296900                Cost
8449     Hokkaido  2024年度                   285500                Cost


In [27]:
# TAHAP 7A PERSIAPAN PROSES AGREGASI DATA

print("Mulai menyiapkan proses agregasi data biaya hidup...")

# Membuat salinan data hasil atribut SAW
df_bh_final = df_bh_saw.copy()

print("Data berhasil disiapkan.")
print(f"Total data awal : {df_bh_final.shape[0]} baris")

Mulai menyiapkan proses agregasi data biaya hidup...
Data berhasil disiapkan.
Total data awal : 235 baris


In [28]:
# TAHAP 7B PERHITUNGAN RATA-RATA BIAYA HIDUP

print("Mulai menghitung rata-rata biaya hidup per prefektur...")

# Menghitung rata-rata biaya hidup berdasarkan prefektur
df_bh_final = df_bh_final.groupby(
    [
        'prefektur_en',
        'kategori_biaya_id',
        'tipe_kriteria_biaya'
    ],
    as_index=False
).agg({
    'biaya_hidup_nominal_yen': 'mean'
})

print("Perhitungan rata-rata berhasil dilakukan.")

Mulai menghitung rata-rata biaya hidup per prefektur...
Perhitungan rata-rata berhasil dilakukan.


In [29]:
# TAHAP 7C PEMBULATAN NILAI HASIL AGREGASI

print("Mulai melakukan pembulatan nilai agregasi...")

# Membulatkan hasil rata-rata menjadi integer
df_bh_final['biaya_hidup_nominal_yen'] = (
    df_bh_final['biaya_hidup_nominal_yen']
    .round()
    .astype('int64')
)

print("Pembulatan nilai berhasil dilakukan.")

Mulai melakukan pembulatan nilai agregasi...
Pembulatan nilai berhasil dilakukan.


In [30]:
# TAHAP 7D PENGURUTAN DATA AKHIR

print("Mulai mengurutkan data akhir...")

# Mengurutkan data berdasarkan nama prefektur
df_bh_final = (
    df_bh_final
    .sort_values(by=['prefektur_en'])
    .reset_index(drop=True)
)

print("Pengurutan data berhasil dilakukan.")

Mulai mengurutkan data akhir...
Pengurutan data berhasil dilakukan.


In [31]:
# TAHAP 7E EKSPOR DATA KE FILE CSV

print("Mulai mengekspor dataset akhir ke file CSV...")

# --- PERUBAHAN PATH TARGET UNTUK DATA FINAL BIAYA HIDUP ---
nama_file_csv_bh = "../data/processed/dataset_biaya_hidup_final.csv"

# Menyimpan dataset ke CSV
df_bh_final.to_csv(
    nama_file_csv_bh,
    index=False,
    encoding='utf-8'
)

print("\n=============================================")
print("        TAHAP 7 SELESAI (BIAYA HIDUP)        ")
print("=============================================")

print("Ekspor file CSV berhasil dilakukan.")
print(f"Nama file / Jalur : {nama_file_csv_bh}")

Mulai mengekspor dataset akhir ke file CSV...

        TAHAP 7 SELESAI (BIAYA HIDUP)        
Ekspor file CSV berhasil dilakukan.
Nama file / Jalur : ../data/processed/dataset_biaya_hidup_final.csv


In [32]:
# TAHAP 7F VERIFIKASI HASIL DATASET FINAL

print("\n=============================================")
print("        TAHAP 7 SELESAI (BIAYA HIDUP)        ")
print("=============================================")

print("Agregasi dan ekspor dataset biaya hidup berhasil dilakukan.")
print(f"Total data akhir : {df_bh_final.shape[0]} baris")

print("\n--- Sampel Dataset Final Biaya Hidup ---")

print(df_bh_final.head(5))


        TAHAP 7 SELESAI (BIAYA HIDUP)        
Agregasi dan ekspor dataset biaya hidup berhasil dilakukan.
Total data akhir : 47 baris

--- Sampel Dataset Final Biaya Hidup ---
  prefektur_en    kategori_biaya_id tipe_kriteria_biaya  \
0        Aichi  Biaya Hidup Bulanan                Cost   
1        Akita  Biaya Hidup Bulanan                Cost   
2       Aomori  Biaya Hidup Bulanan                Cost   
3        Chiba  Biaya Hidup Bulanan                Cost   
4        Ehime  Biaya Hidup Bulanan                Cost   

   biaya_hidup_nominal_yen  
0                   294920  
1                   267240  
2                   254200  
3                   312620  
4                   241760  


In [33]:
print(df_bh_final.columns.tolist())

['prefektur_en', 'kategori_biaya_id', 'tipe_kriteria_biaya', 'biaya_hidup_nominal_yen']
